In [1]:
# Part A — Creating the Date Spine
# Q1. Load the store dataset
import pandas as pd
import numpy as np

STORE_PATH = "store_transactions_sparse.csv"
ENERGY_PATH = "energy_usage_hourly.csv"
store = pd.read_csv(STORE_PATH,parse_dates=["date"])

print("Rows:", len(store))
print("Minimum date:", store["date"].min())
print("Maximum date:", store["date"].max())
print("="*50)
calendar_days = (store["date"].max() - store["date"].min()).days + 1
missing_days = calendar_days - len(store)
print("Calendar days:", calendar_days)
print("Difference:", missing_days)

Rows: 405
Minimum date: 2023-01-02 00:00:00
Maximum date: 2024-02-29 00:00:00
Calendar days: 424
Difference: 19


In [2]:
# Q2. Build a complete daily date spine
spine = pd.date_range(start=store["date"].min(),end=store["date"].max(),freq="D")
store_spined = (store.set_index("date").reindex(spine))
store_spined.index.name = "date"

print(store_spined.shape)
print("="*50)
print(store_spined.head(30))
print("="*30)
print(store_spined.isnull().sum())

(424, 2)
            transactions   revenue
date                              
2023-01-02         788.0  15680.68
2023-01-03         799.0  15184.39
2023-01-04         722.0  10945.52
2023-01-05         862.0  14243.50
2023-01-06         909.0  14807.19
2023-01-07        1039.0  17291.33
2023-01-08         998.0  14855.74
2023-01-09         824.0  12292.29
2023-01-10         777.0  13126.67
2023-01-11         806.0  13787.06
2023-01-12         845.0  15868.05
2023-01-13         958.0  16781.45
2023-01-14        1093.0  22005.53
2023-01-15         930.0  14186.13
2023-01-16         759.0  12071.54
2023-01-17         863.0  15852.28
2023-01-18         921.0  16200.40
2023-01-19         869.0  14358.39
2023-01-20         901.0  17248.96
2023-01-21        1121.0  22011.54
2023-01-22         979.0  19537.24
2023-01-23         846.0  14717.00
2023-01-24         904.0  16250.72
2023-01-25         883.0  16188.42
2023-01-26         878.0  15882.33
2023-01-27         969.0  18913.31
2023-01-28 

In [3]:
# Q3. Missing store days — 0 or interpolation?
store_spined["transactions"] = (store_spined["transactions"].fillna(0))
store_spined["revenue"] = (store_spined["revenue"].fillna(0))

# Store closed
#       ↓
# No transactions
#       ↓
# transactions = 0
# revenue = 0

In [4]:
# Q4. Add was_closed

store_spined = (store.set_index("date").reindex(spine))
store_spined.index.name = "date"
store_spined["was_closed"] = (store_spined["revenue"].isna())
store_spined["transactions"] = (store_spined["transactions"].fillna(0))
store_spined["revenue"] = (store_spined["revenue"].fillna(0))
print(store_spined["was_closed"].value_counts())
print(store_spined[store_spined["was_closed"] == True])

was_closed
False    405
True      19
Name: count, dtype: int64
            transactions  revenue  was_closed
date                                         
2023-04-04           0.0      0.0        True
2023-05-15           0.0      0.0        True
2023-05-24           0.0      0.0        True
2023-07-28           0.0      0.0        True
2023-08-01           0.0      0.0        True
2023-08-19           0.0      0.0        True
2023-08-24           0.0      0.0        True
2023-09-09           0.0      0.0        True
2023-09-23           0.0      0.0        True
2023-10-25           0.0      0.0        True
2023-11-23           0.0      0.0        True
2023-11-24           0.0      0.0        True
2023-12-25           0.0      0.0        True
2024-01-01           0.0      0.0        True
2024-01-02           0.0      0.0        True
2024-01-12           0.0      0.0        True
2024-01-14           0.0      0.0        True
2024-01-16           0.0      0.0        True
2024-01-22       

In [5]:
# Q5. Energy hourly spine
energy = pd.read_csv(ENERGY_PATH,parse_dates=["timestamp"])
energy = energy.sort_values("timestamp")
hourly_spine = pd.date_range(start=energy["timestamp"].min(),end=energy["timestamp"].max(),freq="h")
energy_spined = (energy.set_index("timestamp").reindex(hourly_spine))
energy_spined.index.name = "timestamp"

print("Expected hours:", len(hourly_spine))
print("Actual rows:", len(energy))
print("Missing hours:", energy_spined["kwh"].isna().sum())

Expected hours: 720
Actual rows: 709
Missing hours: 11


In [6]:
energy_spined["kwh"] = (energy_spined["kwh"].interpolate(method="time"))

In [7]:
# Part B — Lag and Lead Functions
# Q6. lag_1 and lag_7
store_spined["lag_1"] = (store_spined["revenue"].shift(1))
store_spined["lag_7"] = (store_spined["revenue"].shift(7))
store_spined.head(9)

,transactions,revenue,was_closed,lag_1,lag_7
date,,,,,
2023-01-02,788.0,15680.68,False,NaN,NaN
2023-01-03,799.0,15184.39,False,15680.68,NaN
2023-01-04,722.0,10945.52,False,15184.39,NaN
2023-01-05,862.0,14243.50,False,10945.52,NaN
2023-01-06,909.0,14807.19,False,14243.50,NaN
2023-01-07,1039.0,17291.33,False,14807.19,NaN
2023-01-08,998.0,14855.74,False,17291.33,NaN
2023-01-09,824.0,12292.29,False,14855.74,15680.68
2023-01-10,777.0,13126.67,False,12292.29,15184.39


In [8]:
# Q7. lead_1
store_spined["lead_1"] = (store_spined["revenue"].shift(-1))

In [9]:
# Q8. Day-over-day change
store_spined["dod_change"] = (store_spined["revenue"]- store_spined["lag_1"])
print(store_spined["dod_change"].head())

print("="*40)
# percentage change
store_spined["dod_pct_change"] = (store_spined["revenue"].pct_change() * 100)
print(store_spined["dod_pct_change"].head())

date
2023-01-02        NaN
2023-01-03    -496.29
2023-01-04   -4238.87
2023-01-05    3297.98
2023-01-06     563.69
Freq: D, Name: dod_change, dtype: float64
date
2023-01-02          NaN
2023-01-03    -3.164978
2023-01-04   -27.915972
2023-01-05    30.130866
2023-01-06     3.957524
Freq: D, Name: dod_pct_change, dtype: float64


In [10]:
# Q9. 365-day lag vs 7-day lag

# I would use a 7-day lag (shift(7)). It allows me to compare each day with the same weekday
# from the previous week, such as this Saturday with last Saturday. This is more useful for 
# identifying weekly seasonality than lag_1, which only compares consecutive days and therefore
# compares different weekdays. A 365-day lag is less useful here because the dataset contains 
# only about 14 months of data, leaving very few observations for year-over-year compariso

store_spined["lag_7"] = (store_spined["revenue"].shift(7))
print(store_spined[["revenue", "lag_7"]].head(25))

             revenue     lag_7
date                          
2023-01-02  15680.68       NaN
2023-01-03  15184.39       NaN
2023-01-04  10945.52       NaN
2023-01-05  14243.50       NaN
2023-01-06  14807.19       NaN
2023-01-07  17291.33       NaN
2023-01-08  14855.74       NaN
2023-01-09  12292.29  15680.68
2023-01-10  13126.67  15184.39
2023-01-11  13787.06  10945.52
2023-01-12  15868.05  14243.50
2023-01-13  16781.45  14807.19
2023-01-14  22005.53  17291.33
2023-01-15  14186.13  14855.74
2023-01-16  12071.54  12292.29
2023-01-17  15852.28  13126.67
2023-01-18  16200.40  13787.06
2023-01-19  14358.39  15868.05
2023-01-20  17248.96  16781.45
2023-01-21  22011.54  22005.53
2023-01-22  19537.24  14186.13
2023-01-23  14717.00  12071.54
2023-01-24  16250.72  15852.28
2023-01-25  16188.42  16200.40
2023-01-26  15882.33  14358.39


In [11]:
# Q10. Record Day
store_spined["lag_14"] = (store_spined["revenue"].shift(14))
print(store_spined["lag_14"].head(20))

print("="*60)
store_spined["record_day"] = (
    (store_spined["revenue"] > store_spined["lag_7"]) &
    (store_spined["revenue"] > store_spined["lag_14"]))

print(store_spined["record_day"].head(20))

date
2023-01-02         NaN
2023-01-03         NaN
2023-01-04         NaN
2023-01-05         NaN
2023-01-06         NaN
2023-01-07         NaN
2023-01-08         NaN
2023-01-09         NaN
2023-01-10         NaN
2023-01-11         NaN
2023-01-12         NaN
2023-01-13         NaN
2023-01-14         NaN
2023-01-15         NaN
2023-01-16    15680.68
2023-01-17    15184.39
2023-01-18    10945.52
2023-01-19    14243.50
2023-01-20    14807.19
2023-01-21    17291.33
Freq: D, Name: lag_14, dtype: float64
date
2023-01-02    False
2023-01-03    False
2023-01-04    False
2023-01-05    False
2023-01-06    False
2023-01-07    False
2023-01-08    False
2023-01-09    False
2023-01-10    False
2023-01-11    False
2023-01-12    False
2023-01-13    False
2023-01-14    False
2023-01-15    False
2023-01-16    False
2023-01-17     True
2023-01-18     True
2023-01-19    False
2023-01-20     True
2023-01-21     True
Freq: D, Name: record_day, dtype: bool


In [12]:
# Part C — Rolling Windows
# Q11. 7-day trailing rolling mean
store_spined["rolling_7_mean"] = (store_spined["revenue"].rolling(window=7).mean())
print(store_spined["rolling_7_mean"].head(10))

date
2023-01-02             NaN
2023-01-03             NaN
2023-01-04             NaN
2023-01-05             NaN
2023-01-06             NaN
2023-01-07             NaN
2023-01-08    14715.478571
2023-01-09    14231.422857
2023-01-10    13937.462857
2023-01-11    14343.397143
Freq: D, Name: rolling_7_mean, dtype: float64


In [13]:
# Q12. Centered rolling mean
store_spined["rolling_7_centered"] = (store_spined["revenue"].rolling(window=7, center=True).mean())
print(store_spined["rolling_7_centered"].head())

date
2023-01-02             NaN
2023-01-03             NaN
2023-01-04             NaN
2023-01-05    14715.478571
2023-01-06    14231.422857
Freq: D, Name: rolling_7_centered, dtype: float64


In [14]:
# Q13. Rolling standard deviation + anomaly flag
store_spined["rolling_7_std"] = (store_spined["revenue"].rolling(window=7).std())
print(store_spined["rolling_7_std"].head(10))

print("="*60)
store_spined["unusual"] = (
abs(
        store_spined["revenue"]- store_spined["rolling_7_mean"])>2 * store_spined["rolling_7_std"]
)

print((store_spined["unusual"]==True).sum())

date
2023-01-02            NaN
2023-01-03            NaN
2023-01-04            NaN
2023-01-05            NaN
2023-01-06            NaN
2023-01-07            NaN
2023-01-08    1925.700650
2023-01-09    2063.572851
2023-01-10    2051.724584
2023-01-11    1590.328218
Freq: D, Name: rolling_7_std, dtype: float64
14


In [15]:
# Q14. min_periods=1
# rolling(7)

store_spined["rolling_7_mean_min1"] = (
    store_spined["revenue"].rolling(window=7, min_periods=1).mean())

print(store_spined["rolling_7_mean_min1"].head(10))

date
2023-01-02    15680.680000
2023-01-03    15432.535000
2023-01-04    13936.863333
2023-01-05    14013.522500
2023-01-06    14172.256000
2023-01-07    14692.101667
2023-01-08    14715.478571
2023-01-09    14231.422857
2023-01-10    13937.462857
2023-01-11    14343.397143
Freq: D, Name: rolling_7_mean_min1, dtype: float64


In [16]:
# Q15. Expanding mean
store_spined["expanding_mean"] = (store_spined["revenue"].expanding().mean())
print(store_spined["expanding_mean"].head(10))

date
2023-01-02    15680.680000
2023-01-03    15432.535000
2023-01-04    13936.863333
2023-01-05    14013.522500
2023-01-06    14172.256000
2023-01-07    14692.101667
2023-01-08    14715.478571
2023-01-09    14412.580000
2023-01-10    14269.701111
2023-01-11    14221.437000
Freq: D, Name: expanding_mean, dtype: float64


In [17]:
# Part D — Aggregating to Different Frequencies
# Q16. Daily → Weekly
weekly = (store_spined.resample("W").agg(
        transactions=("transactions", "sum"),
        revenue=("revenue", "sum")))
weekly.head(10)

,transactions,revenue
date,,
2023-01-08,6117.0,103008.35
2023-01-15,6233.0,108047.18
2023-01-22,6413.0,117280.35
2023-01-29,6516.0,118372.03
2023-02-05,6340.0,112646.62
2023-02-12,6434.0,114535.85
2023-02-19,6363.0,112104.06
2023-02-26,6712.0,119786.44
2023-03-05,6455.0,119074.80


In [18]:
# Q17. Average Transaction Value
monthly = (store_spined.resample("MS").agg(
        revenue=("revenue", "sum"),
        transactions=("transactions", "sum")))
monthly["average_transaction_value"] = (monthly["revenue"]/ monthly["transactions"])
monthly.head(10)

,revenue,transactions,average_transaction_value
date,,,
2023-01-01,474183.38,26900.0,17.627635
2023-02-01,461513.98,25872.0,17.838357
2023-03-01,524963.33,29159.0,18.003475
2023-04-01,518893.22,28423.0,18.256103
2023-05-01,522260.61,28360.0,18.415395
2023-06-01,548136.22,29856.0,18.359332
2023-07-01,552375.99,30326.0,18.214601
2023-08-01,505388.02,28551.0,17.701237
2023-09-01,540126.82,29596.0,18.249994


In [19]:
# Q18. Energy Hourly → Daily → Weekly
# Daily Total
energy_daily = (energy_spined["kwh"].resample("D").sum())
print(energy_daily.head())
print("="*60)
# Weekly Total
energy_weekly = (energy_spined["kwh"].resample("W").sum())
print(energy_weekly.head())
print("="*60)
# Daily → Hourly
hourly_from_daily = (energy_daily.resample("h").asfreq())
hourly_from_daily.interpolate()
print(hourly_from_daily.head())

timestamp
2024-03-01    62.069
2024-03-02    51.795
2024-03-03    50.721
2024-03-04    62.880
2024-03-05    62.527
Freq: D, Name: kwh, dtype: float64
timestamp
2024-03-03    164.585
2024-03-10    415.150
2024-03-17    414.789
2024-03-24    411.572
2024-03-31    360.125
Freq: W-SUN, Name: kwh, dtype: float64
timestamp
2024-03-01 00:00:00    62.069
2024-03-01 01:00:00       NaN
2024-03-01 02:00:00       NaN
2024-03-01 03:00:00       NaN
2024-03-01 04:00:00       NaN
Freq: h, Name: kwh, dtype: float64


In [20]:
# Q19. Average Revenue by Weekday
weekday_avg = (store_spined.groupby(store_spined.index.dayofweek)["revenue"].mean())

weekday_avg.index = [
    "Monday",
    "Tuesday",
    "Wednesday",
    "Thursday",
    "Friday",
    "Saturday",
    "Sunday"
]

weekday_avg = weekday_avg.reset_index()
weekday_avg.columns = [
    "weekday",
    "average_revenue"
]

print(weekday_avg)

     weekday  average_revenue
0     Monday     15573.160328
1    Tuesday     16036.700164
2  Wednesday     16809.885410
3   Thursday     16720.768033
4     Friday     17900.058167
5   Saturday     20426.670333
6     Sunday     19808.142167


In [21]:
# Q20. Monthly Revenue — Two Methods
# Method A — Direct
monthly_direct = (store_spined["revenue"].resample("MS").sum())
print(monthly_direct.head())
print("="*60)

# Method B — Weekly first
weekly_revenue = (store_spined["revenue"].resample("W").sum())
monthly_from_weekly = (weekly_revenue.resample("MS").sum())
print(monthly_from_weekly.head())

date
2023-01-01    474183.38
2023-02-01    461513.98
2023-03-01    524963.33
2023-04-01    518893.22
2023-05-01    522260.61
Freq: MS, Name: revenue, dtype: float64
date
2023-01-01    446707.91
2023-02-01    459072.97
2023-03-01    479508.02
2023-04-01    594265.01
2023-05-01    473253.75
Freq: MS, Name: revenue, dtype: float64


In [22]:
# Part E — Q21 Mini Integration Challenge
import pandas as pd
import numpy as np

# ============================================================
# 1. LOAD DATA
# ============================================================

STORE_PATH = "store_transactions_sparse.csv"
store = pd.read_csv(
    STORE_PATH,
    parse_dates=["date"])

store = store.sort_values("date")


# ============================================================
# 2. CREATE COMPLETE DAILY DATE SPINE
# ============================================================

date_spine = pd.date_range(start=store["date"].min(),end=store["date"].max(),freq="D")
store_clean = (store.set_index("date").reindex(date_spine))
store_clean.index.name = "date"

# ============================================================
# 3. IDENTIFY CLOSED / MISSING DAYS
# ============================================================

store_clean["was_closed"] = (store_clean["revenue"].isna())
print(store_clean["was_closed"].head())
print("="*70)
# ============================================================
# 4. FILL CLOSED DAYS WITH ZERO
# ============================================================
store_clean["transactions"] = (store_clean["transactions"].fillna(0))
store_clean["revenue"] = (store_clean["revenue"].fillna(0))

# ============================================================
# 5. 7-DAY TRAILING ROLLING AVERAGE
# ============================================================

store_clean["rolling_7d_avg_revenue"] = (store_clean["revenue"].rolling(window=7).mean())
print(store_clean["rolling_7d_avg_revenue"].head(20))
print("="*70)

# ============================================================
# 6. RESAMPLE DAILY DATA TO WEEKLY
# ============================================================
weekly = (store_clean.resample("W").agg(
        total_revenue=("revenue", "sum"),
        total_transactions=("transactions", "sum"),
        average_daily_revenue=("revenue", "mean")))
print(weekly.head())
print("="*70)

# ============================================================
# 7. 4-WEEK ROLLING AVERAGE OF WEEKLY REVENUE
# ============================================================
weekly["rolling_4w_avg_revenue"] = (weekly["total_revenue"].rolling(window=4).mean())
print(weekly["rolling_4w_avg_revenue"].head())
print("="*70)

# ============================================================
# 8. WEEK-OVER-WEEK PERCENT CHANGE
# ============================================================
weekly["wow_pct_change"] = (weekly["total_revenue"].pct_change()* 100)
print(weekly["wow_pct_change"].head())
print("="*70)

# ============================================================
# 9. RESET INDEX
# ============================================================
weekly = weekly.reset_index()

# ============================================================
# 10. DISPLAY FINAL TABLE
# ============================================================
print(weekly.to_string(index=False))

date
2023-01-02    False
2023-01-03    False
2023-01-04    False
2023-01-05    False
2023-01-06    False
Freq: D, Name: was_closed, dtype: bool
date
2023-01-02             NaN
2023-01-03             NaN
2023-01-04             NaN
2023-01-05             NaN
2023-01-06             NaN
2023-01-07             NaN
2023-01-08    14715.478571
2023-01-09    14231.422857
2023-01-10    13937.462857
2023-01-11    14343.397143
2023-01-12    14575.475714
2023-01-13    14857.512857
2023-01-14    15530.970000
2023-01-15    15435.311429
2023-01-16    15403.775714
2023-01-17    15793.148571
2023-01-18    16137.911429
2023-01-19    15922.245714
2023-01-20    15989.032857
2023-01-21    15989.891429
Freq: D, Name: rolling_7d_avg_revenue, dtype: float64
            total_revenue  total_transactions  average_daily_revenue
date                                                                
2023-01-08      103008.35              6117.0           14715.478571
2023-01-15      108047.18              6233.0     